# TP 2 — Analyse de la complexité temporelle
### Algorithmique avancée et complexité — Master IA et Master RSD · Chapitre 2

| | |
|---|---|
| **Cours & TD** | Dr. Benyamina Ahmed |
| **TP** | Mahkouka Mohamed Fouad |
| **Durée** | 1 séance |

---
## 🎯 Objectif

Mettre en pratique le chapitre 2 en trois parties, comme dans la fiche de TP :

| Partie | Fiche | But |
|---|---|---|
| 1 | TP 1 — Mesure expérimentale | implémenter 4 algorithmes et mesurer leurs temps d'exécution |
| 2 | TP 2 — Visualisation des complexités | tracer les courbes théoriques $1$, $\log_2 n$, $n$, $n\log_2 n$, $n^2$ |
| 3 | TP 3 — Mini-projet | comparer 4 algorithmes de tri, de l'implémentation à la conclusion |

Dans chaque partie : ✍️ vous **programmez** les fonctions, ✅ une cellule **vérifie** qu'elles sont justes, 📈 les cellules de **mesure et de tracé** sont fournies, puis vous complétez quelques lignes ✍️.

## 🖥️ Environnement

Langage : Python 3 · Outils : `time`, `matplotlib`. Notez votre **machine** (CPU, RAM) et votre **OS** :

> ✍️ **Machine :** … · **OS :** …

> 💡 Pendant les mesures, fermez les applications inutiles. L'affichage et la génération des données ne sont **jamais** inclus dans le temps mesuré.

---
# Outils

**Exécutez cette cellule en premier.** Vous n'avez rien à y modifier.

| Fonction | Rôle |
|---|---|
| `verifier(nom, obtenu, attendu)` | affiche ✅ ou ❌ |
| `chrono(f, *args)` | temps **moyen** de `f(*args)` sur **5 exécutions** |
| `pente(tailles, valeurs)` | pente en log-log : ≈ 0 → $\log n$, ≈ 1 → $n$, un peu plus de 1 → $n\log n$, ≈ 2 → $n^2$ |
| `afficher_tableau`, `tracer`, `sauver_csv` | tableau, graphique log-log (PNG), fichier CSV |

In [ ]:
def verifier(nom, obtenu, attendu):
    print(("✅" if obtenu == attendu else "❌"), nom, "| obtenu :", obtenu, "| attendu :", attendu)


import csv
import math
import random
import time
from statistics import linear_regression
import matplotlib.pyplot as plt


def chrono(f, *args, repetitions=5):
    # Temps moyen (secondes) de f(*args) sur 5 exécutions.
    total = 0.0
    for _ in range(repetitions):
        start = time.perf_counter()
        f(*args)
        total += time.perf_counter() - start
    return total / repetitions


def pente(sizes, values):
    # Pente de la droite en log-log, calculée sur les points mesurés (None ignorés).
    pts = [(n, v) for n, v in zip(sizes, values) if v]
    lx = [math.log10(n) for n, _ in pts]
    ly = [math.log10(v) for _, v in pts]
    return round(linear_regression(lx, ly).slope, 2)


def afficher_tableau(sizes, results):
    # results = {nom: [valeur ou None]}
    names = list(results)
    print(f"{'n':>10} | " + " | ".join(f"{c:>20}" for c in names))
    print("-" * (13 + 23 * len(names)))
    for i, n in enumerate(sizes):
        row = [results[c][i] if i < len(results[c]) else None for c in names]
        cells = [f"{v:>20.3e}" if v is not None else f"{'—':>20}" for v in row]
        print(f"{n:>10,} | " + " | ".join(cells))


def tracer(titre, sizes, results, ylabel="temps (s)", fichier=None):
    # Courbes en échelle log-log ; sauvegarde en PNG si un nom de fichier est donné.
    plt.figure(figsize=(8, 5))
    for name, values in results.items():
        pts = [(n, v) for n, v in zip(sizes, values) if v]
        if pts:
            plt.loglog([p[0] for p in pts], [p[1] for p in pts], "o-", label=name)
    plt.xlabel("taille de l'entrée n (log)")
    plt.ylabel(ylabel)
    plt.title(titre)
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    if fichier:
        plt.savefig(fichier, dpi=120, bbox_inches="tight")
    plt.show()


def sauver_csv(fichier, sizes, results):
    with open(fichier, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["n"] + list(results))
        for i, n in enumerate(sizes):
            w.writerow([n] + [results[c][i] if i < len(results[c]) else None for c in results])
    print("Tableau enregistré :", fichier)


print("Outils prêts ✅")

---
# Partie 1 — Mesure expérimentale de la complexité

**But :** implémenter quatre algorithmes simples et mesurer leurs temps d'exécution pour différentes tailles $n$.

| # | Algorithme | Complexité théorique |
|---|---|---|
| 1 | Recherche linéaire (tableau non trié) | $O(n)$ |
| 2 | Recherche binaire (tableau **trié**) | $O(\log n)$ |
| 3 | Tri par insertion | $O(n^2)$ |
| 4 | Tri fusion (merge sort) | $O(n \log n)$ |

> 💡 Conseil de la fiche : la même **interface** pour chaque fonction. Les recherches prennent `(T, x)` et renvoient l'**indice** de `x` (ou `-1`). Les tris prennent `T` et renvoient une **nouvelle liste triée** (sans modifier `T`, sans utiliser `sorted`).

## 1.1 Recherche linéaire
Parcourir `T` de gauche à droite ; renvoyer l'indice de la première case égale à `x`, ou `-1`.

In [ ]:
def linear_search(T, x):
    for i in range(len(T)):
        if T[i] == x:
            return i
    return -1

verifier("trouvé", linear_search([7, 3, 9, 1], 9), 2)
verifier("absent", linear_search([7, 3, 9, 1], 5), -1)
verifier("liste vide", linear_search([], 5), -1)

## 1.2 Recherche binaire
Dans un tableau **trié** : regarder la case du milieu ; si `x` est plus grand, chercher à droite, sinon à gauche. On s'arrête quand la zone de recherche est vide.

```
bas ← 0 ; haut ← n - 1
tant que bas ≤ haut
    milieu ← (bas + haut) // 2
    si T[milieu] = x alors retourner milieu
    sinon si T[milieu] < x alors bas ← milieu + 1
    sinon haut ← milieu - 1
retourner -1
```

In [ ]:
def binary_search(T, x):
    low, high = 0, len(T) - 1
    while low <= high:
        mid = (low + high) // 2
        if T[mid] == x:
            return mid
        elif T[mid] < x:
            low = mid + 1
        else:
            high = mid - 1
    return -1

T = [1, 3, 5, 7, 9, 11]
verifier("x = 1", binary_search(T, 1), 0)
verifier("x = 11", binary_search(T, 11), 5)
verifier("x = 7", binary_search(T, 7), 3)
verifier("absent", binary_search(T, 4), -1)

## 1.3 Tri par insertion
Pour chaque élément à partir du deuxième : le retirer (`cle`), décaler vers la droite les éléments plus grands, puis l'insérer à sa place.

```
pour i = 1 à n - 1
    cle ← T[i] ; j ← i - 1
    tant que j ≥ 0 et T[j] > cle
        T[j + 1] ← T[j] ; j ← j - 1
    T[j + 1] ← cle
```

In [ ]:
def insertion_sort(T):
    T = T[:]          # copie : la liste d'origine n'est pas modifiée
    for i in range(1, len(T)):
        key = T[i]
        j = i - 1
        while j >= 0 and T[j] > key:
            T[j + 1] = T[j]
            j -= 1
        T[j + 1] = key
    return T

random.seed(0)
data = [random.randint(0, 100) for _ in range(50)]
verifier("50 entiers triés", insertion_sort(data) == sorted(data), True)
verifier("liste vide", insertion_sort([]), [])
verifier("déjà triée", insertion_sort([1, 2, 3]), [1, 2, 3])

## 1.4 Tri fusion
1. `merge(L, R)` **fusionne** deux listes triées en une liste triée : comparer les deux premiers éléments, prendre le plus petit, recommencer ; à la fin, ajouter le reste.
2. `merge_sort(T)` : si `T` a 0 ou 1 élément, il est trié ; sinon couper en deux moitiés, trier chaque moitié (récursion), puis les fusionner.

In [ ]:
def merge(L, R):
    result = []
    i = j = 0
    while i < len(L) and j < len(R):
        if L[i] <= R[j]:
            result.append(L[i]); i += 1
        else:
            result.append(R[j]); j += 1
    result.extend(L[i:])
    result.extend(R[j:])
    return result

def merge_sort(T):
    if len(T) <= 1:
        return T[:]
    m = len(T) // 2
    return merge(merge_sort(T[:m]), merge_sort(T[m:]))

verifier("merge", merge([1, 4, 9], [2, 3, 10, 11]), [1, 2, 3, 4, 9, 10, 11])
verifier("50 entiers triés", merge_sort(data) == sorted(data), True)
verifier("liste vide", merge_sort([]), [])

## 1.5 Mesures

Méthodologie de la fiche : pour chaque $n \in \{10, 100, \dots, 10^6\}$ et chaque algorithme, générer les données, exécuter **5 fois**, garder le **temps moyen**.

- Recherches : **pire cas**, `x` absent (`x = -1`). La recherche binaire travaille sur la version **triée** de `T` (le tri est fait **avant** la mesure).
- ⚠️ En Python, le tri par insertion est limité à $n \le 10\,000$ (pour $10^5$ il faudrait plusieurs minutes par mesure) et le tri fusion à $n \le 100\,000$. Les cases non mesurées valent `None` et s'affichent « — ».

✍️ Complétez la boucle : la recherche linéaire est faite, ajoutez les trois autres algorithmes.

In [ ]:
sizes = [10, 100, 1_000, 10_000, 100_000, 1_000_000]
results1 = {"Recherche linéaire": [], "Recherche binaire": [],
            "Tri par insertion": [], "Tri fusion": []}

random.seed(1)
for n in sizes:
    T = [random.randint(0, 10 * n) for _ in range(n)]
    T_sorted = sorted(T)
    results1["Recherche linéaire"].append(chrono(linear_search, T, -1))
    results1["Recherche binaire"].append(chrono(binary_search, T_sorted, -1))
    results1["Tri par insertion"].append(chrono(insertion_sort, T) if n <= 10_000 else None)
    results1["Tri fusion"].append(chrono(merge_sort, T) if n <= 100_000 else None)

afficher_tableau(sizes, results1)

In [ ]:
for name, values in results1.items():
    if len(values) == len(sizes):
        print(f"{name:<22} pente ≈ {pente(sizes[2:], values[2:])}")   # à partir de n = 1000
tracer("Partie 1 — temps moyens (log-log)", sizes, results1, fichier="tp2_partie1.png")
sauver_csv("resultats_tp2_partie1.csv", sizes, results1)

### Comparaison avec les complexités théoriques

| Algorithme | Complexité théorique | Pente mesurée | Comportement observé | Conforme ? |
|---|---|---|---|---|
| Recherche linéaire | $O(n)$ | ≈ 1 | croissance linéaire | ✅ |
| Recherche binaire | $O(\log n)$ | ≈ 0,1 | croissance très lente, presque plate | ✅ |
| Tri par insertion | $O(n^2)$ | ≈ 2 | croissance quadratique | ✅ |
| Tri fusion | $O(n \log n)$ | ≈ 1 à 1,2 | entre linéaire et quadratique | ✅ |

Les pentes exactes dépendent de la machine.

> ✍️ **Conclusion (2 lignes) :** les mesures confirment $O(\log n) < O(n) < O(n \log n) < O(n^2)$. Les constantes et la machine changent les temps, mais pas l'ordre de grandeur ; dès $n \ge 10^4$ les écarts deviennent très importants.

---
# Partie 2 — Visualisation des complexités

**Objectif :** tracer les courbes **théoriques** des complexités courantes en fonction de $n$ (échelle logarithmique) et les comparer.

| Fonction | Nom |
|---|---|
| $O(1) = 1$ | constante |
| $O(\log n) = \log_2 n$ | logarithmique |
| $O(n) = n$ | linéaire |
| $O(n \log n) = n \log_2 n$ | quasi-linéaire |
| $O(n^2) = n^2$ | quadratique |

✍️ Pour $n = 10, 10^2, \dots, 10^6$, calculez la valeur de chaque fonction et rangez-la dans la liste correspondante de `values`. Utilisez `math.log2`.

In [ ]:
ns = [10**k for k in range(1, 7)]
values = {"O(1)": [], "O(log n)": [], "O(n)": [], "O(n log n)": [], "O(n²)": []}

for n in ns:
    values["O(1)"].append(1)
    values["O(log n)"].append(math.log2(n))
    values["O(n)"].append(n)
    values["O(n log n)"].append(n * math.log2(n))
    values["O(n²)"].append(n ** 2)

verifier("O(log n) pour n = 10", round(values["O(log n)"][0], 2) if values["O(log n)"] else None, 3.32)
verifier("O(n log n) pour n = 10⁶", round(values["O(n log n)"][-1], 2) if values["O(n log n)"] else None, 19931568.57)

In [ ]:
# Tableau des valeurs théoriques, graphique (PNG) et fichier CSV
print(f"{'n':>9} | " + " | ".join(f"{k:>14}" for k in values))
for i, n in enumerate(ns):
    print(f"{n:>9,} | " + " | ".join(f"{values[k][i]:>14,.2f}" if values[k] else f"{'—':>14}" for k in values))

tracer("Comparaison des complexités théoriques", ns, values,
       ylabel="valeur (relative)", fichier="tp2_complexites.png")
sauver_csv("tp2_valeurs.csv", ns, values)

### Question 3 en code
« À partir de quelle valeur de $n$, $O(n^2)$ devient-elle beaucoup plus grande que $O(n)$ ? »
On fixe « beaucoup plus grande » = **au moins 100 fois** plus grande. ✍️ Écrivez une boucle `while` qui trouve le premier $n$ tel que $n^2 \ge 100 \cdot n$.

In [ ]:
n = 1
while n ** 2 < 100 * n:
    n += 1

verifier("premier n", n, 100)

> ✍️ **Q1. Quelle fonction croît le plus lentement ?** $O(1)$ ne croît pas ; parmi les autres, $\log_2 n$ croît le plus lentement (presque plat).

> ✍️ **Q2. Quelle fonction croît le plus rapidement ?** $n^2$ (pente ≈ 2 en log-log).

> ✍️ **Q4. Comment l'échelle log-log aide-t-elle à comparer les complexités ?** les valeurs de $1$ à $10^{12}$ tiennent sur un seul graphique ; chaque $n^k$ devient une droite de pente $k$, et un facteur constant devient un simple décalage vertical.

---
# Partie 3 — Mini-projet : tri d'un tableau de $n$ entiers aléatoires

**Objectif :** comparer plusieurs algorithmes de tri en temps d'exécution et valider leurs complexités théoriques, en suivant toutes les étapes : implémentation → mesure → comparaison → conclusion.

| Algorithme | Idée | Temps | Espace |
|---|---|---|---|
| Tri par insertion | insère chaque élément à sa place dans la partie triée | $O(n^2)$ | $O(1)$ |
| Tri fusion | divise puis fusionne les sous-tableaux | $O(n\log n)$ | $O(n)$ |
| Tri rapide (quick sort) | partitionne autour d'un pivot | $O(n\log n)$ en moyenne, $O(n^2)$ au pire | $O(\log n)$ en moyenne |
| Tri par tas (heap sort) | utilise un tas binaire | $O(n\log n)$ | $O(1)$ |

Le tri par insertion et le tri fusion sont déjà écrits (Partie 1).

## 3.1 Tri rapide
✍️ Choisir comme pivot l'élément du **milieu**. Construire trois listes : les éléments **plus petits** que le pivot, **égaux**, **plus grands**. Renvoyer `tri(plus petits) + égaux + tri(plus grands)`.

In [ ]:
def quick_sort(T):
    if len(T) <= 1:
        return T[:]
    pivot = T[len(T) // 2]
    smaller = [x for x in T if x < pivot]
    equal = [x for x in T if x == pivot]
    larger = [x for x in T if x > pivot]
    return quick_sort(smaller) + equal + quick_sort(larger)

verifier("50 entiers triés", quick_sort(data) == sorted(data), True)
verifier("avec doublons", quick_sort([3, 1, 3, 2, 1]), [1, 1, 2, 3, 3])

## 3.2 Tri par tas *(fourni : lisez-le et exécutez)*
Un **tas max** est un arbre binaire rangé dans un tableau : les enfants de la case `i` sont `2i + 1` et `2i + 2`, et chaque parent est plus grand que ses enfants. Le plus grand élément est donc en `T[0]`. On l'échange avec le dernier, on réduit le tas d'une case, et on fait « redescendre » la nouvelle racine (`sift_down`).

In [ ]:
def sift_down(T, i, end):
    # fait descendre T[i] tant qu'un enfant (avant end) est plus grand
    while 2 * i + 1 < end:
        child = 2 * i + 1
        if child + 1 < end and T[child + 1] > T[child]:
            child += 1
        if T[i] >= T[child]:
            return
        T[i], T[child] = T[child], T[i]
        i = child

def heap_sort(T):
    T = T[:]
    n = len(T)
    for i in range(n // 2 - 1, -1, -1):     # construire le tas max
        sift_down(T, i, n)
    for end in range(n - 1, 0, -1):         # extraire le maximum, un par un
        T[0], T[end] = T[end], T[0]
        sift_down(T, 0, end)
    return T

verifier("50 entiers triés", heap_sort(data) == sorted(data), True)

## 3.3 Protocole de mesure
Pour chaque $n \in \{10^2, 10^3, 10^4, 10^5\}$ : générer **5 tableaux aléatoires**, exécuter chaque algorithme sur chaque tableau, garder le **temps moyen** (`time.perf_counter()`). Le tri par insertion s'arrête à $n = 10^4$.

✍️ Complétez la colonne **« Meilleur »** : pour chaque $n$, le nom de l'algorithme le plus rapide.

In [ ]:
sorts = {"Insertion": insertion_sort, "Fusion": merge_sort, "Rapide": quick_sort, "Tas": heap_sort}
sizes3 = [100, 1_000, 10_000, 100_000]
results3 = {name: [] for name in sorts}

random.seed(3)
for n in sizes3:
    arrays = [[random.randint(0, 10 * n) for _ in range(n)] for _ in range(5)]
    for name, f in sorts.items():
        if name == "Insertion" and n > 10_000:
            results3[name].append(None)
        else:
            results3[name].append(sum(chrono(f, A, repetitions=1) for A in arrays) / 5)

afficher_tableau(sizes3, results3)

In [ ]:
best = []
for i, n in enumerate(sizes3):
    measured = {name: results3[name][i] for name in results3 if results3[name][i] is not None}
    best.append(min(measured, key=measured.get))

print("Meilleur :", best)

In [ ]:
for name, values in results3.items():
    print(f"{name:<10} pente ≈ {pente(sizes3, values)}")
tracer("Partie 3 — algorithmes de tri (log-log)", sizes3, results3, fichier="tp2_partie3.png")
sauver_csv("resultats_tp2_partie3.csv", sizes3, results3)

## 3.4 Analyse et comparaison

| Algorithme | Complexité théorique | Pente mesurée | Comportement observé | Conforme ? |
|---|---|---|---|---|
| Insertion | $O(n^2)$ | ≈ 2 | courbe très raide, impraticable dès $n \approx 10^5$ | ✅ |
| Fusion | $O(n\log n)$ | ≈ 1,1 | croissance modérée, stable | ✅ |
| Rapide (moyen) | $O(n\log n)$ | ≈ 1,1 | souvent le plus rapide sur données aléatoires | ✅ |
| Tas | $O(n\log n)$ | ≈ 1,1 | proche de la fusion, un peu plus lent en Python | ✅ |

Le classement entre fusion, rapide et tas peut changer d'une machine à l'autre : leurs courbes sont parallèles, seules les **constantes** diffèrent.

> ✍️ **Conclusion argumentée (3 à 5 lignes) :** les mesures confirment $O(n^2) \gg O(n\log n)$ : le tri par insertion devient impraticable pour de grandes tailles, alors que les trois tris en $n \log n$ restent rapides. Entre eux, l'écart ne vient que des constantes et de l'implémentation. En pire cas (pivot mal choisi) le tri rapide peut devenir $O(n^2)$, mais sur des données aléatoires il reste $O(n\log n)$. Le choix dépend de la taille des données et de la mémoire disponible (la fusion utilise $O(n)$ de mémoire en plus).

---
# 📤 Travail à rendre

Livrables demandés par la fiche :

- ce notebook complété (il remplace le fichier `.py`) : **Restart & Run All** sans erreur, toutes les vérifications ✅ ;
- les tableaux `resultats_tp2_partie1.csv`, `tp2_valeurs.csv`, `resultats_tp2_partie3.csv` ;
- les graphiques `tp2_partie1.png`, `tp2_complexites.png`, `tp2_partie3.png` ;
- les analyses et conclusions ✍️ (comparaison théorie / expérimentation).

```bash
git add tp2_complexite/
git commit -m "TP2 : analyse de la complexité temporelle"
git push
```

## 🎯 À retenir

- Les mesures expérimentales **valident (ou infirment)** les prédictions théoriques.
- Les constantes et la machine influencent les temps mesurés, **pas l'ordre de grandeur**.
- En log-log, des droites parallèles indiquent la même complexité ; la **pente** donne le degré.
- $O(1) < O(\log n) < O(n) < O(n\log n) < O(n^2)$.

### ➡️ Prochaine séance : **TP 3 — Arbres binaires de recherche**